In [1]:
import pprint
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from typing_extensions import TypedDict
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy
import os

from dotenv import load_dotenv

load_dotenv("C:\\Users\\socgen\\ML\\agentic_ai_and_ops\\langchain_day5\\.env")

True

In [2]:
model_gr_lamma = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=1000, temperature=0.0)


model_or_paid_gpt_luna_pro = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=100, temperature=0.0)

model_or_free_nvidia = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


model_or_free = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


model_ollama = init_chat_model("ollama:gemma4:latest",
                                max_tokens=200, 
                                temperature=0.0)

In [3]:
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")

class CustomerComplaint(BaseModel):
    """A customer complaint about a product or service."""
    issue_type: Literal["product", "service", "shipping", "billing"] = Field(description="The type of issue")
    severity: Literal["low", "medium", "high"] = Field(description="The severity of the complaint")
    description: str = Field(description="Brief description of the complaint")


In [12]:

agent = create_agent(
    model=model_or_free_nvidia,
    tools=[],
    response_format=ToolStrategy(Union[ProductReview, CustomerComplaint])
)


In [15]:

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Not so Great Shoes: 2 out of 5 stars. Fast shipping, expensive and poor quality. I want to complain about the product and the service quality'"}]
})

result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

ProductReview(rating=2, sentiment='negative', key_points=['fast shipping', 'expensive', 'poor quality', 'product complaint', 'service quality'])

In [14]:
result

{'messages': [HumanMessage(content="Analyze this review: 'Not so Great Shoes: 2 out of 5 stars. Fast shipping, expensive and poor quality. I want to complain about the product and the billing'", additional_kwargs={}, response_metadata={}, id='f0b85f92-66aa-451b-b7be-c68b73dfcd4b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants me to analyze a review. The review says: "Not so Great Shoes: 2 out of 5 stars. Fast shipping, expensive and poor quality. I want to complain about the product and the billing."\n\nI need to analyze this review. The user mentions "I want to complain about the product and the billing." That suggests they want to file a complaint. However, the instruction is to "Analyze this review". I have two tools: ProductReview and CustomerComplaint. The ProductReview tool is for analyzing a product review, extracting key points, rating, sentiment. The CustomerComplaint tool is for creating a customer complaint with description, issue_type, sev

In [9]:
result["structured_response"]

ProductReview(rating=2, sentiment='negative', key_points=['fast shipping', 'expensive', 'poor quality'])